In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
! pip install fasttext torch_geometric networkx
import torch
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.utils import to_networkx
import matplotlib.pyplot as plt
import pickle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.1 MB/s eta 0:00:00
  Using cached pybind11-2.11.1-py3-none-any.whl (227 kB)
  Created wheel for fasttext: filename=fasttext-0.9.2-cp310-cp310-linux_x86_64.whl size=4199773 sha256=56064bfa6fe5a8af570370cc9ae646d0f1883500393066377d9c336b7ef23623
  Stored in directory: /root/.cache/pip/wheels/a5/13/75/f811c84a8ab36eedbaef977a6a58a98990e8e0f1967f98f394
Successfully built fasttext


In [3]:
import os
import pandas as pd
import numpy as np

In [4]:
workDir = os.path.join(os.getcwd(), "drive/MyDrive/Colab Notebooks/Graph Sentiment Analysis")

In [5]:
os.listdir(workDir)

['graph.ipynb',
 'Twitter_Data.csv',
 'pre_dataset.pkl',
 'fasttext_model.bin',
 'index.ipynb']

### Getting Corpus of words

In [6]:
with open( os.path.join(workDir,"pre_dataset.pkl") , "rb") as file:
    df = pickle.load(file)

corpus = list ( df["preprocessed_text"] )
words = [word for sent in corpus for word in sent.split()]
# getting only unique words ------
words = list(set(words))

In [7]:
len(words)

86072

### Loading Fasttext Model

In [8]:
import fasttext
from fasttext.FastText import load_model

In [9]:
model = load_model(os.path.join(workDir,"fasttext_model.bin"))

In [10]:
model[words[0]]

array([ 7.98221231e-02, -3.20777178e-01, -1.30774245e-01, -1.62547454e-01,
        1.55648425e-01,  1.73799351e-01,  4.11681496e-02, -1.29693002e-01,
        7.83629268e-02,  3.15741658e-01,  3.13857198e-02,  4.01092887e-01,
        3.01687062e-01, -2.44792193e-01, -1.19427949e-01, -1.23418935e-01,
       -1.77458510e-01,  1.93120435e-01,  2.30039075e-01, -1.96995839e-01,
        2.09646448e-01, -1.29627034e-01,  2.42223144e-01, -7.54655376e-02,
       -1.47610098e-01, -1.10598719e-02, -5.34571856e-02, -1.96467996e-01,
        2.86737364e-02,  1.48696065e-01,  8.43370184e-02,  2.34540403e-01,
       -1.83926182e-04, -3.45648736e-01, -6.37014955e-02,  2.64750700e-02,
        3.06836724e-01, -3.16487476e-02,  3.15215625e-02,  3.18418555e-02,
       -1.13223419e-01,  1.70895442e-01, -3.17215510e-02, -3.26834142e-01,
        3.75426531e-01,  1.65489595e-02, -9.82769132e-02, -7.29615465e-02,
       -5.15198288e-03, -1.02410428e-01, -9.48004052e-02,  6.22697435e-02,
        3.06303889e-01,  

### Making Graph

In [11]:
# vertices matrix
x = torch.tensor( np.array ( [ model[word] for word in words ] , dtype = float), dtype= torch.float)

In [12]:
x.shape

torch.Size([86072, 64])

In [13]:
print(f"No of words = {x.shape[0]} and Dimension of each word is {x.shape[1]} ")

No of words = 86072 and Dimension of each word is 64 


In [14]:
print( x[23].shape)

torch.Size([64])


In [15]:
from nltk.util import ngrams

In [36]:
temp = list( [ngrams(c.split(),3, pad_right = True, pad_left = True) for c in corpus if c is not None ] )
len(temp) == len(corpus)

True

In [38]:
edge_list = []
for temp1 in temp:
  for gram in temp1:
    left, mid, *right = gram
    if mid == None :
      continue

    mid_idx = words.index(mid)

    if left is not None :
      left_idx = words.index(left)
      edge_list.append([left_idx, mid_idx])
      edge_list.append([mid_idx, left_idx])


    if right is not None :
      right_idx = [words.index(r) for r in right if r is not None]

      for r_idx in right_idx :
        edge_list.append([mid_idx, r_idx])
        edge_list.append([r_idx,mid_idx])

KeyboardInterrupt: ignored

In [ ]:
len(edge_list)

In [ ]:
edge_index = torch.tensor(np.array(edge_list, dtype = float) , dtype= torch.long)

In [ ]:
data_obj = Data(x=x, edge_index= edge_index.t().contiguous())

In [ ]:
g = to_networkx(data_obj)
g.draw()